In [3]:
from google.colab import drive; drive.mount('/content/drive')
import os, json, datetime, requests, pandas as pd
OUT = "/content/drive/MyDrive/esci/n2"; os.makedirs(OUT, exist_ok=True)

COUNTRIES = {"LKA": "LKR", "PAK": "PKR", "USA": "USD"}
INDICATORS = {"ppp_private": "PA.NUS.PRVT.PP",   # used: PPP for household consumption
              "ppp_gdp": "PA.NUS.PPP",           # reference only
              "cpi": "FP.CPI.TOTL",              # inflation index (2010 = 100)
              "fx_official_avg": "PA.NUS.FCRF"}  # yearly average official exchange rate (reference)

import time
def wb(indicator):
    out, urls = {}, []
    for iso in COUNTRIES:
        url = (f"https://api.worldbank.org/v2/country/{iso}/indicator/{indicator}"
               f"?format=json&date=2015:2026&per_page=100&source=2")
        urls.append(url)
        for attempt in range(4):
            try:
                payload = requests.get(url, timeout=120).json(); break
            except Exception as e:
                print(f"  {indicator} {iso}: try {attempt+1} failed ({type(e).__name__}), retrying...")
                time.sleep(10 * (attempt + 1))
        else:
            raise RuntimeError(f"World Bank API not answering for {indicator} {iso}. Try again later.")
        for row in (payload[1] or []) if len(payload) > 1 else []:
            if row.get("value") is not None:
                out.setdefault(row["countryiso3code"], {})[int(row["date"])] = float(row["value"])
    return out, urls[0].replace(f"/country/{list(COUNTRIES)[0]}/", f"/country/{';'.join(COUNTRIES)}/")

data, urls = {}, {}
for name, code in INDICATORS.items():
    data[name], urls[name] = wb(code)
    print(f"{name:16s}", {c: (max(v), round(v[max(v)], 3)) for c, v in data[name].items()})

def update_ppp(iso, us="USA"):
    ppp, cpi = data["ppp_private"], data["cpi"]
    base = max(ppp[iso])
    newer = [y for y in set(cpi.get(iso, {})) & set(cpi.get(us, {})) if y > base]
    if newer and base in cpi[iso] and base in cpi[us]:
        t = max(newer)
        ic, iu = cpi[iso][t] / cpi[iso][base], cpi[us][t] / cpi[us][base]
        applied = True
    else:                                  # PPP already as new as the newest CPI -> use it as it is
        t, ic, iu, applied = base, 1.0, 1.0, False
    return {"ppp_base_year": base, "ppp_base": ppp[iso][base], "cpi_year": t, "update_applied": applied,
            "inflation_factor_country": round(ic, 4), "inflation_factor_us": round(iu, 4),
            "ppp_updated": round(ppp[iso][base] * ic / iu, 4)}

fx = requests.get("https://open.er-api.com/v6/latest/USD", timeout=30).json()
assert fx.get("result") == "success", fx

snap = {"created_at": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "method": "PPP_now = PPP_base * (CPI_c(t)/CPI_c(base)) / (CPI_US(t)/CPI_US(base))",
        "sources": {"ppp_private": urls["ppp_private"], "cpi": urls["cpi"],
                    "fx": "https://open.er-api.com/v6/latest/USD"},
        "fx_as_of": fx.get("time_last_update_utc"), "countries": {}}
rows = []
for iso, cur in COUNTRIES.items():
    if iso == "USA": continue
    u = update_ppp(iso)
    fx_now = fx["rates"][cur]
    u.update({"currency": cur, "fx_per_usd_now": fx_now,
              "price_level_ratio": round(u["ppp_updated"] / fx_now, 4),   # < 1 = cheaper than the USA
              "ppp_gdp_latest": {str(k): v for k, v in sorted(data["ppp_gdp"].get(iso, {}).items())[-1:]}})
    snap["countries"][iso] = u
    rows.append({"country": iso, **{k: u[k] for k in ["ppp_base_year", "ppp_base", "cpi_year", "update_applied",
             "inflation_factor_country", "inflation_factor_us", "ppp_updated", "fx_per_usd_now", "price_level_ratio"]}})
print("\n", pd.DataFrame(rows).to_string(index=False))

day = datetime.date.today().isoformat()
for name in (f"rates_snapshot_{day}.json", "rates_snapshot.json"):
    json.dump(snap, open(f"{OUT}/{name}", "w"), indent=2)
json.dump({k: {c: {str(y): v for y, v in s.items()} for c, s in d.items()} for k, d in data.items()},
          open(f"{OUT}/wb_raw_{day}.json", "w"), indent=2)
print("\nsaved:", f"{OUT}/rates_snapshot_{day}.json")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ppp_private      {'LKA': (2025, 81.132), 'PAK': (2025, 65.396), 'USA': (2025, 1.0)}
ppp_gdp          {'LKA': (2025, 88.214), 'PAK': (2025, 67.837), 'USA': (2025, 1.0)}
cpi              {'LKA': (2025, 292.339), 'PAK': (2025, 400.518), 'USA': (2024, 143.857)}
fx_official_avg  {'LKA': (2025, 300.852), 'PAK': (2025, 281.143), 'USA': (2025, 1.0)}

 country  ppp_base_year  ppp_base  cpi_year  update_applied  inflation_factor_country  inflation_factor_us  ppp_updated  fx_per_usd_now  price_level_ratio
    LKA           2025 81.132322      2025           False                       1.0                  1.0      81.1323      331.714106             0.2446
    PAK           2025 65.396146      2025           False                       1.0                  1.0      65.3961      276.999315             0.2361

saved: /content/drive/MyDrive/esci/n2/rates_snapshot_2026-10-2